# Glass Box AI: asking the FDA database questions

One small agent: a system prompt, two reference documents (skills), and three tools. Ask a question in plain English and this notebook shows the answer **and** the work behind it: every query the agent ran and what came back.

The agent is defined once in `agent/`. This notebook is the AWS Strands runtime; LibreChat runs the same prompt, skills, and tools for live demos.

Run the cells top to bottom. Needs `LLM_MODEL_ID` and an API key in `.env`.

In [ ]:
import os

from IPython.display import Markdown, display

from agent.strands_agent import SKILLS_DIR, SYSTEM_PROMPT_PATH, ask, build_agent

print('Model:', os.environ['LLM_MODEL_ID'])

## What the agent is made of

In [ ]:
display(Markdown('### System prompt\n\n' + SYSTEM_PROMPT_PATH.read_text()))

In [ ]:
for skill in sorted(SKILLS_DIR.iterdir()):
    text = (skill / 'SKILL.md').read_text()
    print(f'{skill.name}: {len(text.split()):,} words  ->  {skill / "SKILL.md"}')

print('\nTools the model can call:', sorted(build_agent().tool_names))

The `skills` tool is Strands' own skill loader: the model sees each skill's name and description, and loads the full text when it needs it. LibreChat has an equivalent loader, its `skill` tool.

## Show your work

`ask_and_show` prints the answer, then the glass-box view: every tool call and every query with its result.

In [ ]:
def ask_and_show(question: str) -> None:
    agent = build_agent()  # fresh conversation and an empty plan each time
    answer = ask(agent, question)
    display(Markdown(f'**Question:** {question}'))
    display(Markdown(answer.text))
    print('\n--- how it got there ---')
    print('Tool calls:', ' -> '.join(answer.tool_calls) or '(none)')
    for number, query in enumerate(answer.queries, start=1):
        print(f'\n[{number}] {query.sql.strip()}\n{query.result}')

## A straightforward question

In [ ]:
ask_and_show('How many Class I recall events were classified in fiscal year 2025?')

## A multi-step question

The agent writes a plan, then works through it.

In [ ]:
ask_and_show(
    'How many firms had a Class I recall event classified in fiscal year 2025 '
    'and also had an OAI inspection result in fiscal year 2025?'
)

## An ambiguous question

"Last year" could be fiscal or calendar, and the data ends in October 2026. A trustworthy agent asks instead of guessing.

In [ ]:
ask_and_show('How many inspections were there last year?')

## A question the data cannot answer

The dataset has no warning letters. The agent should say so plainly.

In [ ]:
ask_and_show('How many warning letters did the FDA issue to drug manufacturers in fiscal year 2025?')

## Check the result before using it

People check the answer: compare the SQL above with what you meant. To measure how often the agent is right and how consistent it is across repeated runs, run the benchmark from a terminal:

```bash
docker compose exec jupyter pytest -m agent
```